##### Setup and Data Acquisition

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, array_contains, explode, size, collect_list, array, lit, avg, round, concat_ws, when, lower, trim, regexp_replace, to_timestamp, expr
from pyspark.sql.types import StructType, StructField, StringType,LongType, IntegerType, FloatType, ArrayType, MapType, BooleanType, TimestampType
import json
import os
import requests

##### Schema Definition and DataFrame Creation

In [2]:

# Function to initialize Spark session
def initialize_spark():
    spark = SparkSession.builder \
        .appName("TMDB Movie Data Processing") \
        .getOrCreate()
    return spark

# Function to define schema for DataFrame
def get_schema():
    return StructType([
        # Movie base fields
        StructField("adult", BooleanType(), True),
        StructField("backdrop_path", StringType(), True),
        StructField("belongs_to_collection", MapType(StringType(), StringType()), True),
        StructField("budget", LongType(), True),
        StructField("genres", ArrayType(MapType(StringType(), StringType())), True),
        StructField("homepage", StringType(), True),
        StructField("id", IntegerType(), True),
        StructField("imdb_id", StringType(), True),
        StructField("original_language", StringType(), True),
        StructField("original_title", StringType(), True),
        StructField("overview", StringType(), True),
        StructField("popularity", FloatType(), True),
        StructField("poster_path", StringType(), True),
        StructField("production_companies", ArrayType(MapType(StringType(), StringType())), True),
        StructField("production_countries", ArrayType(MapType(StringType(), StringType())), True),
        StructField("release_date", StringType(), True),
        StructField("revenue", LongType(), True),
        StructField("runtime", IntegerType(), True),
        StructField("spoken_languages", ArrayType(MapType(StringType(), StringType())), True),
        StructField("status", StringType(), True),
        StructField("tagline", StringType(), True),
        StructField("title", StringType(), True),
        StructField("video", BooleanType(), True),
        StructField("vote_average", FloatType(), True),
        StructField("vote_count", IntegerType(), True),

        # Nested data for credits
        StructField("credits", StructType([
            StructField("id", IntegerType(), True),
            StructField("cast", ArrayType(StructType([
                StructField("adult", BooleanType(), True),
                StructField("gender", IntegerType(), True),
                StructField("id", IntegerType(), True),
                StructField("known_for_department", StringType(), True),
                StructField("name", StringType(), True),
                StructField("original_name", StringType(), True),
                StructField("popularity", FloatType(), True),
                StructField("profile_path", StringType(), True),
                StructField("cast_id", IntegerType(), True),
                StructField("character", StringType(), True),
                StructField("credit_id", StringType(), True),
                StructField("order", IntegerType(), True)
            ]), True)),
            StructField("crew", ArrayType(StructType([
                StructField("adult", BooleanType(), True),
                StructField("gender", IntegerType(), True),
                StructField("id", IntegerType(), True),
                StructField("known_for_department", StringType(), True),
                StructField("name", StringType(), True),
                StructField("original_name", StringType(), True),
                StructField("popularity", FloatType(), True),
                StructField("profile_path", StringType(), True),
                StructField("credit_id", StringType(), True),
                StructField("department", StringType(), True),
                StructField("job", StringType(), True)
            ]), True))
        ]), True),

        # Nested data for reviews
        StructField("reviews", StructType([
            StructField("id", IntegerType(), True),
            StructField("page", IntegerType(), True),
            StructField("results", ArrayType(StructType([
                StructField("author", StringType(), True),
                StructField("author_details", StructType([
                    StructField("name", StringType(), True),
                    StructField("username", StringType(), True),
                    StructField("avatar_path", StringType(), True),
                    StructField("rating", FloatType(), True)
                ]), True),
                StructField("content", StringType(), True),
                StructField("created_at", StringType(), True),
                StructField("id", StringType(), True),
                StructField("updated_at", StringType(), True),
                StructField("url", StringType(), True)
            ]), True)),
            StructField("total_pages", IntegerType(), True),
            StructField("total_results", IntegerType(), True)
        ]), True)
    ])

# Function to fetch movie data from the TMDB API
def fetch_movie_data(movie_ids, api_token):
    headers = {
        "Authorization": f"Bearer {api_token}"
    }
    movies_data = []
    for movie_id in movie_ids:
        try:
            url = f"https://api.themoviedb.org/3/movie/{movie_id}?append_to_response=reviews,credits"
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                movies_data.append(response.json())
                print(f"Successfully fetched movie ID: {movie_id}")
            else:
                print(f"Failed to fetch movie ID: {movie_id}, Status code: {response.status_code}")
        except Exception as e:
            print(f"Error fetching movie ID {movie_id}: {e}")
    return movies_data

# Shared function for both scripts
def load_from_parquet(file_path, spark):
    if os.path.exists(file_path):
        df = spark.read.parquet(file_path)
        print(f"Loaded data from {file_path}")
        return df
    else:
        print(f"File not found: {file_path}")
        return None

# Function to save DataFrame to Parquet
def save_to_parquet(df, file_path):
    df.write.mode("overwrite").parquet(file_path)
    print(f"Data saved to {file_path}")



In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession
import os

# Main process to fetch and save movie data
def process_movie_data():
    spark = initialize_spark()
    parquet_file_path = "./dataset/raw/raw_tmdb_movies.parquet"
    schema = get_schema()

    # Check if the Parquet file exists
    raw_movies_df = load_from_parquet(parquet_file_path, spark)

    if raw_movies_df is None:
        # Fetch movie data if the Parquet file doesn't exist
        api_token = os.getenv("TMDB_API_KEY")
        if not api_token:
            print("API token not found.")
            exit(1)

        movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397,
                     420818, 24428, 168259, 99861, 284054, 12445,
                     181808, 330457, 351286, 109445, 321612, 260513]

        movies_data = fetch_movie_data(movie_ids, api_token)

        # If data was retrieved, create a DataFrame
        if movies_data:
            raw_movies_df = spark.createDataFrame(movies_data, schema)
            save_to_parquet(raw_movies_df, parquet_file_path)
        else:
            print("No movie data retrieved. Cannot create DataFrame.")
            exit(1)


# Function to extract movie columns from nested data
def get_movie_columns(df):
    """Extract additional columns from nested credits and reviews data"""
    
    # Extract cast_size
    df = df.withColumn("cast_size", 
                       F.size(F.col("credits.cast")))
    
    # Extract crew_size
    df = df.withColumn("crew_size", 
                       F.size(F.col("credits.crew")))
    
    # Extract directors
    df = df.withColumn("directors", 
                       F.expr("transform(filter(credits.crew, c -> c.job = 'Director'), d -> d.name)"))
    
    # Extract cast names
    df = df.withColumn("cast", 
                       F.expr("transform(credits.cast, c -> c.name)"))
    
    # Calculate average rating from reviews
    df = df.withColumn("rating", 
                       F.when(F.size(F.expr("filter(reviews.results, r -> r.author_details.rating is not null)")) > 0,
                            F.round(F.expr("""
                                aggregate(
                                    filter(reviews.results, r -> r.author_details.rating is not null), 
                                    cast(0 as double), 
                                    (acc, r) -> acc + cast(r.author_details.rating as double), 
                                    sum -> sum / size(filter(reviews.results, r -> r.author_details.rating is not null))
                                )
                            """), 2)
                           ).otherwise(None))
    
    return df

# Function to clean movie data
def clean_movie_data(df, pipe_columns=None, collection_col=None, columns_to_drop=None):
    """Clean movie data with essential transformations"""
    
    # Process pipe-separated columns
    if pipe_columns:
        for col_name in pipe_columns:
            if col_name in df.columns:
                df = df.withColumn(col_name, 
                                  F.concat_ws("|", F.expr(f"transform({col_name}, x -> x.name)")))
    
    # Process collection name
    if collection_col and collection_col in df.columns:
        df = df.withColumn(collection_col, 
                         F.col(f"{collection_col}.name"))
    
    # Drop irrelevant columns
    if columns_to_drop:
        df = df.drop(*columns_to_drop)
    
    print("Data cleaning completed.")
    return df


def handle_incorrect_data(extracted_df, numeric_columns = None, money_columns=None, placeholders= None):
    
    # Replace zeros with null
    for col_name in numeric_columns:
        extracted_df = extracted_df.withColumn(
            col_name,
            F.when((F.col(col_name).isNotNull()) & (F.col(col_name) != 0), F.col(col_name)).otherwise(None)
        )

    # Convert money columns to millions
    for col_name in money_columns:
        if col_name in extracted_df.columns:
            extracted_df = extracted_df.withColumn(col_name, F.col(col_name) / 1000000)

    # Rename money columns to include _musd suffix
    for old_name in money_columns:
        new_name = f"{old_name}_musd"
        extracted_df = extracted_df.withColumnRenamed(old_name, new_name)

    # Process string columns to replace placeholders with NaN
    string_cols = [f.name for f in extracted_df.schema.fields if isinstance(f.dataType, StringType)]

    for col_name in string_cols:
        # Replace placeholder values with NaN
        extracted_df = extracted_df.withColumn(
            col_name,
            F.when(~F.col(col_name).isin(placeholders) & F.col(col_name).isNotNull(),
                  F.col(col_name)).otherwise(None) )  
     # Save the final processed data
    extracted_df.write.mode("overwrite").parquet("./dataset/processed/tmdb_movies_processed.parquet")
    print("Processed movie data saved to tmdb_movies_processed.parquet")

    return extracted_df


#### Data Cleaning and Preprocessing

In [4]:
# Main processing function
def main():
    # Initialize Spark
    spark = initialize_spark()
    
    # Use process_movie_data to get or fetch raw data
    parquet_file_path = "./dataset/raw/raw_tmdb_movies.parquet"
    schema = get_schema()

    # Check if the Parquet file exists
    raw_movies_df = load_from_parquet(parquet_file_path, spark)

    if raw_movies_df is None:
        # Fetch movie data if the Parquet file doesn't exist
        api_token = os.getenv("TMDB_API_KEY")
        if not api_token:
            print("API token not found.")
            exit(1)

        movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397,
                     420818, 24428, 168259, 99861, 284054, 12445,
                     181808, 330457, 351286, 109445, 321612, 260513]

        movies_data = fetch_movie_data(movie_ids, api_token)

        # If data was retrieved, create a DataFrame
        if movies_data:
            raw_movies_df = spark.createDataFrame(movies_data, schema)
            save_to_parquet(raw_movies_df, parquet_file_path)
        else:
            print("No movie data retrieved. Cannot create DataFrame.")
            exit(1)
            
    # Process the data
    # First extract additional columns
    extracted_df = get_movie_columns(raw_movies_df)

    # Then clean and transform
    extracted_df = clean_movie_data(
        df=extracted_df,
        pipe_columns=['genres', 'spoken_languages', 'production_countries', 'production_companies'],
        collection_col='belongs_to_collection',
        columns_to_drop=['adult', 'imdb_id', 'original_title', 'video', 'homepage', 'reviews', 'credits', 'backdrop_path']
    )

    # Define column categories for final processing
    numeric_columns = ['budget', 'revenue', 'id', 'popularity', 'vote_average', 'vote_count', 'rating', 'crew_size', 'cast_size', 'runtime']
    money_columns = ['budget', 'revenue']
    placeholders = ['no data', 'none', 'n/a', '']
    
    # Handle incorrect data using your custom function
    processed_df = handle_incorrect_data(extracted_df, 
                                        numeric_columns=numeric_columns,
                                        money_columns=money_columns,
                                        placeholders=placeholders)
    
  
    column_order = ['id', 'title', 'tagline', 'release_date', 'genres', 'belongs_to_collection',
               'original_language', 'budget_musd', 'revenue_musd', 'production_companies',
               'production_countries', 'vote_count', 'vote_average', 'popularity', 'runtime',
               'overview', 'spoken_languages', 'poster_path', 'cast', 'cast_size', 'director', 'crew_size']

# Get existing columns from the DataFrame
    existing_columns = set(processed_df.columns)

# Create the final column order:
# First add columns that exist in both the desired order and DataFrame
# Then add any remaining columns from the DataFrame
    final_columns = [col for col in column_order if col in existing_columns] + \
                [col for col in processed_df.columns if col not in set(column_order)]

# Reorder in one operation
    processed_df = processed_df.select(*final_columns)


    processed_df = processed_df.withColumn(
             "profit", col("revenue_musd") - col("budget_musd")
         ).withColumn(
            "roi", when(col("budget_musd") > 0, col("revenue_musd") / col("budget_musd")).otherwise(None)
        )


    # Show sample of the processed data
    print("Sample of processed data:")
    processed_df.show(5, truncate=True)

    processed_df.createOrReplaceTempView("movies")
    
    def get_movie_rankings_sql(spark):
        queries = {
            "Highest Revenue": """
                SELECT * FROM movies ORDER BY revenue_musd DESC
            """,
            "Highest Budget": """
                SELECT * FROM movies ORDER BY budget_musd DESC
            """,
            "Highest Profit": """
                SELECT * FROM movies ORDER BY profit DESC
            """,
            "Lowest Profit": """
                SELECT * FROM movies ORDER BY profit ASC
            """,
            "Highest ROI (Budget ≥ 10M)": """
                SELECT * FROM movies WHERE budget_musd >= 10 ORDER BY roi DESC
            """,
            "Lowest ROI (Budget ≥ 10M)": """
                SELECT * FROM movies WHERE budget_musd >= 10 ORDER BY roi ASC
            """,
            "Most Voted Movies": """
                SELECT * FROM movies ORDER BY vote_count DESC
            """,
            "Highest Rated Movies (≥10 votes)": """
                SELECT * FROM movies WHERE vote_count >= 10 ORDER BY rating DESC
            """,
            "Lowest Rated Movies (≥10 votes)": """
                SELECT * FROM movies WHERE vote_count >= 10 ORDER BY rating ASC
            """,
            "Most Popular Movies": """
                SELECT * FROM movies ORDER BY popularity DESC
            """,
            "Shortest Runtime Movies": """
                SELECT * FROM movies ORDER BY runtime ASC
            """
        }

        rankings = {}
        for name, query in queries.items():
            rankings[name] = spark.sql(query)
        
        return rankings

    all_sql_rankings = get_movie_rankings_sql(spark)

# top 5 highest rated movies with ≥10 votes
    all_sql_rankings["Highest Rated Movies (≥10 votes)"].show(5)

    # # Fix column name if cast is present
    # if 'cast' in processed_df.columns:
    #     processed_df = processed_df.withColumnRenamed("cast", "cast_list")

    # # Add rating column if missing
    # if 'rating' not in processed_df.columns and 'vote_average' in processed_df.columns:
    #     processed_df = processed_df.withColumn("rating", col("vote_average"))

    # processed_df = processed_df.withColumn("genres_str", concat_ws("|", col("genres")))

    # processed_df.createOrReplaceTempView("movies")


    # spark.sql("""
    #     SELECT *
    #     FROM movies
    #     WHERE genres_str LIKE '%Science Fiction%'
    #     AND genres_str LIKE '%Action%'
    #     AND cast_list LIKE '%Bruce Willis%'
    #     ORDER BY rating DESC
    # """).show(truncate=False)



    
    # Stop Spark session
    spark.stop()

if __name__ == "__main__":
        main()
        

Loaded data from ./dataset/raw/raw_tmdb_movies.parquet
Data cleaning completed.
Processed movie data saved to tmdb_movies_processed.parquet
Sample of processed data:
+------+--------------------+--------------------+------------+--------------------+---------------------+-----------------+-----------+------------+--------------------+--------------------+----------+------------+----------+-------+--------------------+--------------------+--------------------+--------------------+---------+---------+--------+--------------------+------+-----------+-----------------+
|    id|               title|             tagline|release_date|              genres|belongs_to_collection|original_language|budget_musd|revenue_musd|production_companies|production_countries|vote_count|vote_average|popularity|runtime|            overview|    spoken_languages|         poster_path|                cast|cast_size|crew_size|  status|           directors|rating|     profit|              roi|
+------+--------------

##### kpi IMplementation

+ Identify the Best/Worst Performing Movies
+  Compare movie franchises (belongs_to_collection) vs. standalone movies in terms of metrics
+ Most Successful Franchises & Directors